In [166]:
import pandas as pd
import os
import re

In [211]:
model_name="tnn_delta"
model_count="TNN_Count"

In [207]:
def count_example():
    counts = {}
    for filename in os.listdir(f'out/{model_name}/df'):
        if filename.endswith('.csv'):
            # 讀取CSV檔案
            df = pd.read_csv(os.path.join(f'out/{model_name}/df', filename))
            # 遍歷每一行
            for index, row in df.iterrows():
                if row[model_count] != 0:
                    # 將對應的Iteration的計數增加1
                    if row['Iteration'] in counts:
                        counts[row['Iteration']] += 1
                    else:
                        counts[row['Iteration']] = 1
    for key in sorted(counts.keys()):
        print('count_{}: {}'.format(key, counts[key]))

In [208]:
count_example()

count_1: 4
count_2: 9
count_3: 15
count_4: 19
count_5: 22
count_6: 24
count_7: 26
count_8: 30


## average size

In [170]:
import os
import re
import pandas as pd


def count_size(directory):
    total = 0
    count = 0
    last_in_iteration = {}
    for root, dirs, files in os.walk(directory):
        for filename in files:
            if filename.endswith('constraint_size.txt'):
                with open(os.path.join(root, filename), 'r') as f:
                    for line in f:
                        numbers = re.findall(r'[-+]?[.]?[\d]+(?:,\d\d\d)*[\.]?\d*(?:[eE][-+]?\d+)?', line)
                        if numbers:
                            total += float(numbers[-1])
                            count += 1
                            iteration = int(numbers[0])
                            last_in_iteration[iteration] = float(numbers[-1])
    average = total / count if count > 0 else 0
    average_last = sum(last_in_iteration.values()) / len(last_in_iteration) if len(last_in_iteration) > 0 else 0
    average_kb=round(average*1024,2)
    average_last_kb=round(average_last*1024,2)


    # 創建一個DataFrame來存儲結果
    df_1 = pd.DataFrame({
        'Average size of the constraint': [average_kb],
    })
    df_2 = pd.DataFrame({
        'Average of the last number in each iteration': [average_last_kb]
    })
        
    return df_1,df_2


In [171]:
# 初始化一個空的DataFrame來存儲所有的結果
results_1 = pd.DataFrame()
results_2 = pd.DataFrame()

# 調用函數並將結果添加到results DataFrame中
for i in range(8):
    df_1,df_2 = count_size(f'exp/mnist_sep_act_m6_9628/queue/test_{model_name}/test_{i}_pixel_forward')
    results_1 = pd.concat([results_1, df_1], axis=1)
    results_2 = pd.concat([results_2, df_2], axis=1)


# 將結果存儲到一個CSV文件中
results_1.to_csv(f'static/{model_name}/avesize_constraint.csv', index=False)
results_2.to_csv(f'static/{model_name}/avesize_last.csv', index=False)

## count time

In [172]:
def count_time(directory):
    # 初始化變數來追蹤總數和數量
    total = 0
    count = 0

    # 初始化字典來追蹤每個Iteration的最後一個數字
    last_in_iteration = {}

    # 遍歷資料夾及其子資料夾中的所有TXT檔案
    for root, dirs, files in os.walk(directory):
        for filename in files:
            if filename.endswith('constraint_time.txt'):
                # 讀取TXT檔案
                with open(os.path.join(root, filename), 'r') as f:
                    # 遍歷每一行
                    for line in f:
                        # 使用正規表達式找出數字
                        numbers = re.findall(r'[-+]?[.]?[\d]+(?:,\d\d\d)*[\.]?\d*(?:[eE][-+]?\d+)?', line)
                        if numbers:
                            # 更新總數和數量
                            total += float(numbers[-1])
                            count += 1
                            # 更新該Iteration的最後一個數字
                            iteration = int(numbers[0])
                            last_in_iteration[iteration] = float(numbers[-1])


    # 計算平均值
    average = total / count if count > 0 else 0
    average_last = sum(last_in_iteration.values()) / len(last_in_iteration) if len(last_in_iteration) > 0 else 0
    average_ms=round(average*1000,2)
    average_last_ms=round(average_last*1000,2)


    df_1 = pd.DataFrame({
        'Average time of the constraint': [average_ms]
    })
    df_2 = pd.DataFrame({
        'Average of the last number in each iteration': [average_last_ms]
    })
    
    return df_1,df_2


In [173]:
results_1 = pd.DataFrame()
results_2 = pd.DataFrame()

# 調用函數並將結果添加到results DataFrame中
for i in range(8):
    df_1,df_2 = count_time(f'exp/mnist_sep_act_m6_9628/queue/test_{model_name}/test_{i}_pixel_forward')
    results_1 = pd.concat([results_1, df_1], axis=1)
    results_2 = pd.concat([results_2, df_2], axis=1)


# 將結果存儲到一個CSV文件中
results_1.to_csv(f'static/{model_name}/avetime_constraint.csv', index=False)
results_2.to_csv(f'static/{model_name}/avestime_last.csv', index=False)

# count smt

In [174]:
def count_smt2(directory):
    # 初始化變數來追蹤smt2檔案數量
    count = 0
    for root, dirs, files in os.walk(directory):
        for filename in files:
            if filename.endswith('.smt2'):
                # 如果檔案是smt2檔案，則增加計數
                count += 1
    count=round(count,2)
    df = pd.DataFrame({
        'Number of smt2 files:': [count]
    })
    
    return df

In [175]:
results = pd.DataFrame()

# 調用函數並將結果添加到results DataFrame中
for i in range(8):
    df= count_smt2(f'exp/mnist_sep_act_m6_9628/queue/test_{model_name}/test_{i}_pixel_forward')
    results = pd.concat([results, df], axis=1)
results.to_csv(f'static/{model_name}/count_smt.csv', index=False)

## count sat smt

In [176]:
def count_sat_smt2(directory):
    # 初始化變數來追蹤smt2檔案數量
    count = 0
    for root, dirs, files in os.walk(directory):
        for filename in files:
            if filename.endswith('_sat.smt2'):
                # 如果檔案是smt2檔案，則增加計數
                count += 1
    count=round(count,2)
    df = pd.DataFrame({
        'Number of sat smt2 files:': [count]
    })
    
    return df

In [177]:
results = pd.DataFrame()

# 調用函數並將結果添加到results DataFrame中
for i in range(8):
    df= count_sat_smt2(f'exp/mnist_sep_act_m6_9628/queue/test_{model_name}/test_{i}_pixel_forward')
    results = pd.concat([results, df], axis=1)
results.to_csv(f'static/{model_name}/count_sat_smt.csv', index=False)

## avg sat smt

In [178]:
def avg_sat_smt2(directory):
    # 初始化變數來追蹤smt2檔案數量和資料夾數量
    file_count = 0
    dir_count = 0
    for root, dirs, files in os.walk(directory):
        smt2_in_current_dir = sum(filename.endswith('_sat.smt2') for filename in files)
        if smt2_in_current_dir > 0:
            # 如果當前資料夾中有smt2檔案，則增加檔案計數和資料夾計數
            file_count += smt2_in_current_dir
            dir_count += 1

    average = file_count / dir_count if dir_count > 0 else 0
    average=round(average,5)
    df = pd.DataFrame({
        'Average number of smt2 files per directory:': [average]
    })
    return df

In [179]:
results = pd.DataFrame()

# 調用函數並將結果添加到results DataFrame中
for i in range(8):
    df= avg_sat_smt2(f'exp/mnist_sep_act_m6_9628/queue/test_{model_name}/test_{i}_pixel_forward')
    results = pd.concat([results, df], axis=1)
results.to_csv(f'static/{model_name}/ave_sat_smt.csv', index=False)

## avg smt

In [180]:
def avg_smt2(directory):
    # 初始化變數來追蹤smt2檔案數量和資料夾數量
    file_count = 0
    dir_count = 0

    for root, dirs, files in os.walk(directory):
        smt2_in_current_dir = sum(filename.endswith('.smt2') for filename in files)
        if smt2_in_current_dir > 0:
            # 如果當前資料夾中有smt2檔案，則增加檔案計數和資料夾計數
            file_count += smt2_in_current_dir
            dir_count += 1


    # 計算平均值
    average = file_count / dir_count if dir_count > 0 else 0
    average=round(average,2)

    df = pd.DataFrame({
        'Average number of smt2 files per directory:': [average]
    })
    return df

In [181]:
results = pd.DataFrame()

# 調用函數並將結果添加到results DataFrame中
for i in range(8):
    df= avg_smt2(f'exp/mnist_sep_act_m6_9628/queue/test_{model_name}/test_{i}_pixel_forward')
    results = pd.concat([results, df], axis=1)
results.to_csv(f'static/{model_name}/ave_smt.csv', index=False)

## time_out

In [186]:
import os
import json


def count_timeouts(directory):
    # 建立一個字典來儲存每個子資料夾的timeout數量
    timeout_dict = {}


    # 獲取所有子資料夾
    subfolders = [f.path for f in os.scandir(directory) if f.is_dir()]


    # 對每個子資料夾進行遍歷
    for subfolder in subfolders:
        timeout_count = 0
        for dirpath, dirs, files in os.walk(subfolder):
            for filename in files:
                if filename == 'stats.json':
                    with open(os.path.join(dirpath, filename)) as f:
                        data = json.load(f)
                        if data['meta']['is_timeout']==True:
                            timeout_count += 1
        timeout_dict[subfolder] = timeout_count


    # 將字典轉換為DataFrame
    df = pd.DataFrame(list(timeout_dict.items()), columns=['Subfolder', 'Timeout_Count'])
    return df
directory = f'exp/mnist_sep_act_m6_9628/queue/test_{model_name}'  # 更換為你的資料夾路徑
timeout_dict = count_timeouts(directory)


# # 打印每個子資料夾的timeout數量
# for subfolder, count in timeout_dict.items():
#     print(f'{subfolder}: {count} timeouts')

timeout_dict.to_csv(f'static/{model_name}/time_out_num.csv', index=False)



## success

In [ ]:


def count_success(directory):
    # 建立一個列表來儲存每個子資料夾和對應的input_name
    result_list = []


    # 獲取所有子資料夾
    subfolders = [f.path for f in os.scandir(directory) if f.is_dir()]


    # 對每個子資料夾進行遍歷
    for subfolder in subfolders:
        for dirpath, dirs, files in os.walk(subfolder):
            for filename in files:
                if filename == 'stats.json':
                    with open(os.path.join(dirpath, filename)) as f:
                        data = json.load(f)
                        if data['meta']['attack_label'] is not None:
                            result_list.append([subfolder, data['meta']['input_name']])
    
    # 將列表轉換為DataFrame
    df = pd.DataFrame(result_list, columns=['Subfolder', 'Input_Name'])
    return df


# 使用函數
directory = f'exp/mnist_sep_act_m6_9628/queue/test_{model_name}' 
df = count_success(directory)


df.to_csv(f'static/{model_name}/example_num.csv', index=False)





In [200]:
def get_smt_files_stats(dir_path):
    all_file_sizes = []
    for root, dirs, files in os.walk(dir_path):
        smt_files = sorted([f for f in files if f.endswith('.smt2')])
        if smt_files:
            sizes = [os.path.getsize(os.path.join(root, f)) for f in smt_files]
            all_file_sizes.append(sizes)
    
    max_count = max(len(sizes) for sizes in all_file_sizes)
    total_sizes = [0]*max_count
    counts = [0]*max_count


    for sizes in all_file_sizes:
        for i in range(len(sizes)):
            total_sizes[i] += sizes[i]
            counts[i] += 1


    avg_sizes = [total_sizes[i] / counts[i] if counts[i] != 0 else 0 for i in range(max_count)]
    
    return avg_sizes






base_dir= f'exp/mnist_sep_act_m6_9628/queue/test_{model_name}' 
dir_names = [f'test_{i}_pixel_forward' for i in range(8)]


for dir_name in dir_names:
    dir_path = f'{base_dir}/{dir_name}'
    print(dir_path)
    avg_sizes, counts = get_smt_files_stats(dir_path)


    print(f'For directory {dir_name}:')
    print('Average sizes: ', avg_sizes)
    print('Counts: ', counts)
    print('\n')

exp/mnist_sep_act_m6_9628/queue/test_cnn/test_0_pixel_forward


ValueError: too many values to unpack (expected 2)

In [197]:
def list_directory_structure(dir_path):
    for root, dirs, files in os.walk(dir_path):
        level = root.replace(dir_path, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print('{}{}/'.format(indent, os.path.basename(root)))
        subindent = ' ' * 4 * (level + 1)
        for f in files:
            print('{}{}'.format(subindent, f))


base_dir= f'exp/mnist_sep_act_m6_9628/queue/test_{model_name}' 
dir_names = [f'test_{i}_pixel_forward' for i in range(8)]
base_dir = os.path.join(base_dir, dir_name)
list_directory_structure(base_dir)





test_0_pixel_forward/
    0/
        mnist_test_0constraint_size.txt
        mnist_test_0constraint_time.txt
        mnist_test_0iteration_count_timeout.txt
        mnist_test_0sat_constraint_memory_usage.txt
        mnist_test_0/
            sat_inputs.npy
            stats.json
            timeouts.csv
            formula/
                1_1_unsat.smt2
                1_2_unsat.smt2
                1_3_unsat.smt2
                1_4_unsat.smt2
                1_5_sat.smt2
                2_1_unsat.smt2
                2_2_unsat.smt2
                2_3_unsat.smt2
                2_4_sat.smt2
                3_1_unsat.smt2
                3_2_unsat.smt2
                3_3_unsat.smt2
                3_4_sat.smt2
                4_1_unsat.smt2
                4_2_unsat.smt2
                4_3_unsat.smt2
                4_4_unsat.smt2
                4_5_unsat.smt2
                4_6_sat.smt2
                5_10_unsat.smt2
                5_11_unsat.smt2
                5_12_sat.smt